In [ ]:
import time
import random
import logging
import pandas as pd
import numpy as np

# ----------------------------
# 1. LOGGING SETUP
# ----------------------------
logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force = True
)
# FORCE file creation
logging.info("Pipeline started")

# ----------------------------
# 2. DATA GENERATOR
# ----------------------------
def generate_data():
    return pd.DataFrame({
        "id": range(50),
        "value": np.random.randint(1, 100, 50)
    })

# ----------------------------
# 3. PIPELINE STEPS
# ----------------------------
def extract():
    logging.info("Extract started")
    time.sleep(random.uniform(0.1, 0.3))
    df = generate_data()
    logging.info("Extract completed")
    return df

def transform(df):
    logging.info("Transform started")
    time.sleep(random.uniform(0.2, 0.5))

    # simulate occasional failure
    if random.random() < 0.3:
        raise Exception("Transformation failed due to bad data")

    df["value"] = df["value"] * 2
    logging.info("Transform completed")
    return df

def load(df):
    logging.info("Load started")
    time.sleep(random.uniform(0.1, 0.2))
    logging.info(f"Load completed with {len(df)} rows")

# ----------------------------
# 4. RUN PIPELINE MULTIPLE TIMES
# ----------------------------
metrics = []

for i in range(10):
    start = time.time()
    status = "SUCCESS"

    try:
        df = extract()
        df = transform(df)
        load(df)
    except Exception as e:
        status = "FAILURE"
        logging.error(str(e))

    latency = round(time.time() - start, 3)

    metrics.append({
        "run_id": i,
        "latency": latency,
        "status": status
    })

# ----------------------------
# 5. SHOW METRICS
# ----------------------------
df_metrics = pd.DataFrame(metrics)

print("\nPipeline Metrics:\n")
print(df_metrics)

# ----------------------------
# 6. SHOW LOGS
# ----------------------------
print("\nSample Logs:\n")
with open("pipeline.log", "r") as f:
    lines = f.readlines()

# show last 10 log lines
for line in lines[-10:]:
    print(line.strip())


Pipeline Metrics:

   run_id  latency   status
0       0    0.677  FAILURE
1       1    0.638  SUCCESS
2       2    0.738  SUCCESS
3       3    0.475  FAILURE
4       4    0.598  SUCCESS
5       5    0.821  SUCCESS
6       6    0.579  FAILURE
7       7    0.594  SUCCESS
8       8    0.512  FAILURE
9       9    0.538  FAILURE

Sample Logs:

2026-04-26 04:33:28,312 - INFO - Load started
2026-04-26 04:33:28,470 - INFO - Load completed with 50 rows
2026-04-26 04:33:28,470 - INFO - Extract started
2026-04-26 04:33:28,572 - INFO - Extract completed
2026-04-26 04:33:28,572 - INFO - Transform started
2026-04-26 04:33:28,982 - ERROR - Transformation failed due to bad data
2026-04-26 04:33:28,982 - INFO - Extract started
2026-04-26 04:33:29,231 - INFO - Extract completed
2026-04-26 04:33:29,232 - INFO - Transform started
2026-04-26 04:33:29,520 - ERROR - Transformation failed due to bad data


In [ ]:
# pip install openai

In [ ]:
from openai import OpenAI

In [ ]:
# ----------------------------
# 1. SET API KEY
# ----------------------------
client = OpenAI(api_key="#######3")

# ----------------------------
# 2. READ LOGS
# ----------------------------
with open("pipeline.log", "r") as f:
    logs = f.read()

In [ ]:
# ----------------------------
# 3. PREPARE PROMPT
# ----------------------------
prompt = f"""
You are an AI Observability assistant.

Analyze the following pipeline logs and metrics.

Logs :
{logs}

Metrics :
{df_metrics.to_string()}

Answer Clearly :
1) What issues do you observe?
2) What is the likey root cause?
3) What recommendations would you give?
"""

In [ ]:
# ----------------------------
# 4. CALL OPENAI
# ----------------------------
response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = [{"role": "user","content": prompt}],
    temperature = 0
)

In [ ]:
# ----------------------------
# 5. PRINT AI RESPONSE
# ----------------------------
print(response.choices[0].message.content)

1) **Issues Observed:**
- Multiple transformation failures due to "bad data" are logged repeatedly (at least 5 distinct transformation failure events).
- Each transformation failure is followed by a successful retry cycle (extract → transform → load).
- Despite retries, some runs still fail overall, as indicated by the metrics where runs 0, 3, 6, 8, and 9 have status FAILURE.
- The latency for failed runs is not consistently higher or lower than successful runs, indicating failures are not due to timeouts or performance issues.
- The last log entry shows an HTTP 401 Unauthorized error when making a POST request to the OpenAI API, indicating an authentication issue unrelated to the pipeline's ETL steps but possibly part of a downstream or monitoring process.

2) **Likely Root Cause:**
- The primary root cause is the presence of "bad data" causing transformation failures. This bad data is not fully filtered or corrected before the transform step, leading to repeated errors.
- The pipelin